# **Preprocessing Mouth**

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


class Config:
    def __init__(self):
        self.root_dir = "/content/dataset1/"
        self.save_root = "/content/output_mouth"

        self.split = "train"
        self.region = "mouth"

        self.model_path = "/content/face_landmarker.task"
        self.detection_threshold = 0.2

        self.gray_fill_value = 128
        self.fail_if_label_missing = True
        self.normalize_output_condition = True
        self.save_grayscale = True
        self.save_ext = ".png"

        self.enable_adaptive_smoothing = True
        self.motion_scale = 20.0

        self.mouth_width_scale = 1.45
        self.mouth_height_scale = 2.5
        self.mouth_vertical_shift = 0.03


class FaceMouthLandmarkDetector:
    def __init__(self, model_path, threshold=0.2):
        self.threshold = threshold

        base_options = python.BaseOptions(model_asset_path=model_path)
        options = vision.FaceLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            num_faces=1,
            min_face_detection_confidence=threshold,
            min_face_presence_confidence=threshold,
            min_tracking_confidence=threshold,
            output_face_blendshapes=False,
            output_facial_transformation_matrixes=False,
        )
        self.landmarker = vision.FaceLandmarker.create_from_options(options)

        self.mouth_indices = [
            61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291,
            78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
            191, 80, 81, 82, 13, 312, 311, 310, 415
        ]

    def detect(self, frame_bgr):
        try:
            h, w = frame_bgr.shape[:2]

            if len(frame_bgr.shape) == 2:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_GRAY2RGB)
            else:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            result = self.landmarker.detect(mp_image)
        except Exception:
            return None

        if not result.face_landmarks:
            return None

        face_landmarks = result.face_landmarks[0]

        def pt(idx):
            return (
                float(face_landmarks[idx].x * w),
                float(face_landmarks[idx].y * h)
            )

        xs = [lm.x * w for lm in face_landmarks]
        ys = [lm.y * h for lm in face_landmarks]
        mouth_points = [pt(idx) for idx in self.mouth_indices]

        return {
            "score": 1.0,
            "bbox": [float(min(xs)), float(min(ys)), float(max(xs)), float(max(ys))],
            "landmarks": {
                "left_eye": pt(33),
                "right_eye": pt(263),
                "nose": pt(1),
                "mouth_left": pt(61),
                "mouth_right": pt(291),
                "mouth_points": mouth_points,
            }
        }

    def close(self):
        self.landmarker.close()


def standardize_condition_name(condition):
    condition = condition.strip().lower()
    mapping = {
        "nightnoglasses": "nightnoglasses",
        "night_no_glasses": "nightnoglasses",
        "night-noglasses": "nightnoglasses",
        "night_noglasses": "nightnoglasses",
        "noglasses": "noglasses",
        "no_glasses": "noglasses",
        "nightglasses": "nightglasses",
        "glasses": "glasses",
        "sunglasses": "sunglasses",
    }
    return mapping.get(condition, condition)


def build_output_condition(condition, cfg):
    if cfg.normalize_output_condition:
        return standardize_condition_name(condition)
    return condition


def label_suffix_for_region(region):
    region = region.lower()
    if region != "mouth":
        raise ValueError("Only mouth supported")
    return "mouth"


def parse_validation_or_test_video_name(video_stem):
    parts = video_stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Invalid filename format: {video_stem}")

    person_id = parts[0]
    state = parts[-1]
    raw_condition = "_".join(parts[1:-1])
    condition = standardize_condition_name(raw_condition)

    return person_id, condition, state


def load_labels(label_path):
    if not os.path.isfile(label_path):
        raise FileNotFoundError(label_path)

    with open(label_path, "r") as f:
        content = f.read().strip()

    return [int(ch) for ch in content if ch.isdigit()]


def get_video_fps(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 30

    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()

    if fps is None or fps <= 0:
        return 30

    return int(round(fps))


def get_fps_fallback_limits(fps):
    fps_int = int(round(fps)) if fps and fps > 0 else 30
    if fps_int == 30:
        return 4, 18
    return 2, 9


def crop_with_padding(img, x1, y1, x2, y2):
    x1 = int(round(x1))
    y1 = int(round(y1))
    x2 = int(round(x2))
    y2 = int(round(y2))

    h, w = img.shape[:2]

    pad_left = max(0, -x1)
    pad_top = max(0, -y1)
    pad_right = max(0, x2 - w)
    pad_bottom = max(0, y2 - h)

    if pad_left or pad_top or pad_right or pad_bottom:
        img = cv2.copyMakeBorder(
            img,
            pad_top,
            pad_bottom,
            pad_left,
            pad_right,
            borderType=cv2.BORDER_REPLICATE
        )

    x1 += pad_left
    x2 += pad_left
    y1 += pad_top
    y2 += pad_top

    return img[y1:y2, x1:x2]


def is_valid_mouth(landmarks):
    points = landmarks.get("mouth_points")
    nose = np.array(landmarks["nose"], dtype=np.float32)

    if not points or len(points) < 4:
        return False

    pts = np.array(points, dtype=np.float32)
    min_xy = pts.min(axis=0)
    max_xy = pts.max(axis=0)

    width = float(max_xy[0] - min_xy[0])
    height = float(max_xy[1] - min_xy[1])
    center = pts.mean(axis=0)
    mouth_to_nose = np.linalg.norm(center - nose)

    if width < 10:
        return False
    if height < 4:
        return False
    if mouth_to_nose < 5:
        return False

    return True


def compute_mouth_box_from_landmarks(landmarks, cfg):
    pts = np.array(landmarks["mouth_points"], dtype=np.float32)

    min_x = pts[:, 0].min()
    max_x = pts[:, 0].max()
    min_y = pts[:, 1].min()
    max_y = pts[:, 1].max()

    mouth_width = max_x - min_x
    mouth_height = max_y - min_y

    cx = (min_x + max_x) / 2.0
    cy = (min_y + max_y) / 2.0

    cy += mouth_height * cfg.mouth_vertical_shift

    crop_w = max(int(round(mouth_width * cfg.mouth_width_scale)), 12)
    crop_h = max(int(round(mouth_height * cfg.mouth_height_scale)), 8)

    x1 = int(round(cx - crop_w / 2.0))
    y1 = int(round(cy - crop_h / 2.0))
    x2 = x1 + crop_w
    y2 = y1 + crop_h

    return [x1, y1, x2, y2]


def smooth_box(prev_box, curr_box, motion_scale=20.0):
    if prev_box is None:
        return [int(round(v)) for v in curr_box]

    prev = np.array(prev_box, dtype=np.float32)
    curr = np.array(curr_box, dtype=np.float32)

    movement = np.linalg.norm(curr - prev)
    motion_factor = min(1.0, movement / motion_scale)

    alpha = 0.7 - 0.4 * motion_factor
    smoothed = alpha * prev + (1.0 - alpha) * curr

    return [int(round(v)) for v in smoothed]


def extract_crop(frame, box, save_grayscale=True):
    if box is None:
        return None

    x1, y1, x2, y2 = box

    if save_grayscale:
        if len(frame.shape) == 2:
            img = frame
        else:
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        if len(frame.shape) == 2:
            img = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
        else:
            img = frame

    crop = crop_with_padding(img, x1, y1, x2, y2)

    if crop.size == 0:
        return None

    return crop


def make_gray_from_box(box, cfg):
    if box is not None:
        x1, y1, x2, y2 = box
        h = max(1, int(round(y2 - y1)))
        w = max(1, int(round(x2 - x1)))

        if cfg.save_grayscale:
            return np.full((h, w), cfg.gray_fill_value, dtype=np.uint8)
        return np.full((h, w, 3), cfg.gray_fill_value, dtype=np.uint8)

    if cfg.save_grayscale:
        return np.full((32, 32), cfg.gray_fill_value, dtype=np.uint8)
    return np.full((32, 32, 3), cfg.gray_fill_value, dtype=np.uint8)


class OnlineMouthCropper:
    def __init__(self, cfg, max_prev_crop, max_prev_position):
        self.cfg = cfg
        self.max_prev_crop = max_prev_crop
        self.max_prev_position = max_prev_position

        self.prev_crop = None
        self.prev_box = None
        self.prev_real_box = None
        self.missing_count = 0

    def step(self, frame, face):
        if face is not None and is_valid_mouth(face["landmarks"]):
            raw_box = compute_mouth_box_from_landmarks(face["landmarks"], self.cfg)

            if self.cfg.enable_adaptive_smoothing:
                box = smooth_box(
                    self.prev_real_box,
                    raw_box,
                    motion_scale=self.cfg.motion_scale
                )
            else:
                box = [int(round(v)) for v in raw_box]

            crop = extract_crop(frame, box, self.cfg.save_grayscale)

            if crop is not None:
                self.prev_crop = crop
                self.prev_box = box
                self.prev_real_box = raw_box
                self.missing_count = 0
                return crop, 1, "detected"

        self.missing_count += 1

        if self.prev_crop is not None and self.missing_count <= self.max_prev_crop:
            return self.prev_crop.copy(), 0, "prev_crop"

        if self.prev_box is not None and self.missing_count <= self.max_prev_position:
            crop = extract_crop(frame, self.prev_box, self.cfg.save_grayscale)
            if crop is not None:
                return crop, 0, "prev_position"

        return make_gray_from_box(self.prev_box, self.cfg), 0, "gray"


def process_video(video_path, label_path, output_dir, save_prefix, region, detector, cfg):
    labels = load_labels(label_path)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    reported_total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = get_video_fps(video_path)

    max_prev_crop, max_prev_position = get_fps_fallback_limits(fps)
    cropper = OnlineMouthCropper(cfg, max_prev_crop, max_prev_position)

    frame_idx = 0
    actual_frames_read = 0

    detected_count = 0
    prev_crop_count = 0
    prev_position_count = 0
    gray_count = 0

    valid_count = 0
    invalid_count = 0

    try:
        while frame_idx < len(labels):
            ret, frame = cap.read()
            if not ret:
                break

            actual_frames_read += 1

            face = detector.detect(frame)
            crop, validity_mask, status = cropper.step(frame, face)

            label = labels[frame_idx]
            name = f"{save_prefix}_{frame_idx:06d}_{validity_mask}_{label}{cfg.save_ext}"
            save_path = os.path.join(output_dir, name)

            ok = cv2.imwrite(save_path, crop)
            if not ok:
                raise RuntimeError(f"Failed to save image: {save_path}")

            if validity_mask == 1:
                valid_count += 1
            else:
                invalid_count += 1

            if status == "detected":
                detected_count += 1
            elif status == "prev_crop":
                prev_crop_count += 1
            elif status == "prev_position":
                prev_position_count += 1
            elif status == "gray":
                gray_count += 1
            else:
                raise ValueError(f"Unknown status: {status}")

            frame_idx += 1

    finally:
        cap.release()

    total = frame_idx if frame_idx > 0 else 1

    print("=" * 80)
    print(f"Video                : {os.path.basename(video_path)}")
    print(f"Region               : {region}")
    print(f"Output folder        : {output_dir}")
    print(f"FPS                  : {fps}")
    print(f"Prev crop limit      : {max_prev_crop}")
    print(f"Prev position limit  : {max_prev_position}")
    print(f"Reported video frames: {reported_total_frames}")
    print(f"Actual frames read   : {actual_frames_read}")
    print(f"Total labels         : {len(labels)}")
    print(f"Frames saved         : {frame_idx}")
    print(f"Valid frames         : {valid_count}")
    print(f"Invalid frames       : {invalid_count}")
    print("-" * 80)
    print(f"Detected normally    : {detected_count} ({detected_count/total:.2%})")
    print(f"Prev crop reused     : {prev_crop_count} ({prev_crop_count/total:.2%})")
    print(f"Prev position        : {prev_position_count} ({prev_position_count/total:.2%})")
    print(f"Gray fallback        : {gray_count} ({gray_count/total:.2%})")
    print("-" * 80)

    missing_from_video = len(labels) - actual_frames_read
    if missing_from_video > 0:
        print(f"Warning              : video ended early by {missing_from_video} frame(s)")
    elif missing_from_video < 0:
        print(f"Warning              : video has {-missing_from_video} extra frame(s)")
    else:
        print("Status               : labels and readable frames matched")
    print("=" * 80)


def process_train(cfg, detector):
    if not os.path.isdir(cfg.root_dir):
        raise FileNotFoundError(f"Root directory not found: {cfg.root_dir}")

    suffix = label_suffix_for_region(cfg.region)

    for person_entry in sorted(os.scandir(cfg.root_dir), key=lambda e: e.name):
        if not person_entry.is_dir():
            continue

        person_id = person_entry.name

        for condition_entry in sorted(os.scandir(person_entry.path), key=lambda e: e.name):
            if not condition_entry.is_dir():
                continue

            raw_condition = condition_entry.name
            out_condition = build_output_condition(raw_condition, cfg)

            for file_entry in sorted(os.scandir(condition_entry.path), key=lambda e: e.name):
                if not file_entry.is_file() or not file_entry.name.lower().endswith(".avi"):
                    continue

                behavior = os.path.splitext(file_entry.name)[0]
                video_path = file_entry.path
                label_path = os.path.join(condition_entry.path, f"{person_id}_{behavior}_{suffix}.txt")

                if not os.path.isfile(label_path):
                    msg = f"Missing train label: {label_path}"
                    if cfg.fail_if_label_missing:
                        raise FileNotFoundError(msg)
                    print(f"[SKIP] {msg}")
                    continue

                output_dir = os.path.join(cfg.save_root, person_id, out_condition, behavior)
                os.makedirs(output_dir, exist_ok=True)

                save_prefix = f"{person_id}_{out_condition}_{behavior}"

                process_video(
                    video_path=video_path,
                    label_path=label_path,
                    output_dir=output_dir,
                    save_prefix=save_prefix,
                    region=cfg.region,
                    detector=detector,
                    cfg=cfg,
                )


def process_validation(cfg, detector):
    if not os.path.isdir(cfg.root_dir):
        raise FileNotFoundError(f"Root directory not found: {cfg.root_dir}")

    suffix = label_suffix_for_region(cfg.region)

    for person_entry in sorted(os.scandir(cfg.root_dir), key=lambda e: e.name):
        if not person_entry.is_dir():
            continue

        folder_person_id = person_entry.name

        for file_entry in sorted(os.scandir(person_entry.path), key=lambda e: e.name):
            if not file_entry.is_file() or not file_entry.name.lower().endswith((".avi", ".mp4")):
                continue

            video_stem = os.path.splitext(file_entry.name)[0]
            person_id, condition, state = parse_validation_or_test_video_name(video_stem)

            if person_id != folder_person_id:
                raise ValueError(f"Subject mismatch: {file_entry.path}")

            label_path = os.path.join(person_entry.path, f"{person_id}_{condition}_mixing_{suffix}.txt")

            if not os.path.isfile(label_path):
                msg = f"Missing validation label: {label_path}"
                if cfg.fail_if_label_missing:
                    raise FileNotFoundError(msg)
                print(f"[SKIP] {msg}")
                continue

            out_condition = build_output_condition(condition, cfg)
            output_dir = os.path.join(cfg.save_root, person_id, out_condition, state)
            os.makedirs(output_dir, exist_ok=True)
            save_prefix = f"{person_id}_{out_condition}_{state}"

            process_video(
                video_path=file_entry.path,
                label_path=label_path,
                output_dir=output_dir,
                save_prefix=save_prefix,
                region=cfg.region,
                detector=detector,
                cfg=cfg,
            )


def process_test(cfg, detector):
    if not os.path.isdir(cfg.root_dir):
        raise FileNotFoundError(f"Root directory not found: {cfg.root_dir}")

    label_root = os.path.join(cfg.root_dir, "test_label_txt", "wh")
    if not os.path.isdir(label_root):
        raise FileNotFoundError(f"Test label folder not found: {label_root}")

    for entry in sorted(os.scandir(cfg.root_dir), key=lambda e: e.name):
        if not entry.is_file() or not entry.name.lower().endswith((".avi", ".mp4")):
            continue

        video_stem = os.path.splitext(entry.name)[0]
        person_id, condition, state = parse_validation_or_test_video_name(video_stem)
        label_path = os.path.join(label_root, f"{person_id}_{condition}_mixing_drowsiness.txt")

        if not os.path.isfile(label_path):
            msg = f"Missing test label: {label_path}"
            if cfg.fail_if_label_missing:
                raise FileNotFoundError(msg)
            print(f"[SKIP] {msg}")
            continue

        out_condition = build_output_condition(condition, cfg)
        output_dir = os.path.join(cfg.save_root, person_id, out_condition, state)
        os.makedirs(output_dir, exist_ok=True)
        save_prefix = f"{person_id}_{out_condition}_{state}"

        process_video(
            video_path=entry.path,
            label_path=label_path,
            output_dir=output_dir,
            save_prefix=save_prefix,
            region=cfg.region,
            detector=detector,
            cfg=cfg,
        )


def run(cfg):
    os.makedirs(cfg.save_root, exist_ok=True)

    detector = FaceMouthLandmarkDetector(
        model_path=cfg.model_path,
        threshold=cfg.detection_threshold,
    )

    try:
        split = cfg.split.lower()
        if split == "train":
            process_train(cfg, detector)
        elif split in {"validation", "val"}:
            process_validation(cfg, detector)
        elif split == "test":
            process_test(cfg, detector)
        else:
            raise ValueError(f"Unsupported split: {cfg.split}")
    finally:
        detector.close()


if __name__ == "__main__":
    cfg = Config()

    cfg.root_dir = "/content/dataset1/"
    cfg.save_root = "/content/output_mouth"

    cfg.split = "train"
    cfg.region = "mouth"

    cfg.model_path = "/content/face_landmarker.task"
    cfg.detection_threshold = 0.2

    cfg.mouth_width_scale = 1.45
    cfg.mouth_height_scale = 2.5
    cfg.mouth_vertical_shift = 0.03

    cfg.enable_adaptive_smoothing = True
    cfg.motion_scale = 20.0

    cfg.gray_fill_value = 128
    cfg.fail_if_label_missing = True
    cfg.normalize_output_condition = True
    cfg.save_grayscale = True

    run(cfg)